## Libraries

In [1]:
import sys
sys.path.append('/hdd4/giuder/Documents/projects/dime')

import faiss
import numpy as np
import pandas as pd
from src.data_loading import CollectionLoader

In [2]:
# src/dime/report.py
import logging

import pandas as pd

from src.config import RUNS_DIR, COLLECTIONS
from src.data_loading import CollectionLoader
from src.evaluate import evaluate, summary as eval_summary

logger = logging.getLogger(__name__)

DEFAULT_MEASURES = ["nDCG@10", "AP", "R@1000", "RR@10"]

In [3]:
def sanity_check(
    collections: str | list[str] = None,
    models: list[str] | None = None,
) -> pd.DataFrame:
    """
    For each (collection, model), check that every query_id in the run
    has at least one relevance judgment in qrels.

    Prints a summary and returns a DataFrame with one row per
    (collection, model) showing counts and any missing query ids.

    Usage:
        sanity_check("dl19")
        sanity_check(["dl19", "dl20"])
        sanity_check(["dl19", "dl20"], models=["contriever"])
    """
    if collections is None:
        collections = list(COLLECTIONS.keys())
    if isinstance(collections, str):
        collections = [collections]

    rows = []
    all_ok = True

    for collection in collections:
        runs_dir = RUNS_DIR / collection
        if not runs_dir.exists():
            logger.warning(f"No runs directory at {runs_dir} — skipping.")
            continue

        qrels      = CollectionLoader(collection).qrels()
        qrels_qids = set(qrels["query_id"].astype(str).unique())

        for path in sorted(runs_dir.glob("*.tsv")):
            if "__" in path.stem:
                continue

            model_name = path.stem
            if models and model_name not in models:
                continue

            run = pd.read_csv(
                path, sep="\t", header=None,
                names=["query_id", "Q0", "doc_id", "rank", "score", "run"],
                dtype={"query_id": str, "doc_id": str},
            )

            run_qids  = set(run["query_id"].unique())
            missing   = sorted(run_qids - qrels_qids)
            ok        = len(missing) == 0
            if not ok:
                all_ok = False

            rows.append({
                "collection":   collection,
                "model":        model_name,
                "run_queries":  len(run_qids),
                "qrel_queries": len(qrels_qids),
                "missing":      len(missing),
                "ok":           "✓" if ok else "✗",
                "missing_ids":  missing,
            })

    if not rows:
        raise ValueError(f"No baseline runs found for collections={collections}.")

    df = pd.DataFrame(rows).set_index(["collection", "model"])

    print(df[["run_queries", "qrel_queries", "missing", "ok"]].to_string())
    print()
    if all_ok:
        print("All queries have relevance judgments ✓")
    else:
        print("Missing query ids:")
        for (collection, model), row in df.iterrows():
            if row["missing_ids"]:
                print(f"  {collection}/{model}: {row['missing_ids']}")
    print()

    return df

## Hyperparams tuning

In [13]:
import pandas as pd
dictCE = pd.read_csv("/hdd4/giuder/Documents/projects/dime/data/other_files/dictCE.tsv", sep="\t", header=None, 
                     names=["query_id", "N/A", "doc_id", "ce_score"])

In [25]:
dictCE['query_id'].unique().shape

(502939,)

In [27]:
dictCE["ce_score"] / 1000

0           1.000
1           0.962
2           0.936
3           0.929
4           0.884
            ...  
50292040    0.176
50292041    0.173
50292042    0.166
50292043    0.164
50292044    0.115
Name: ce_score, Length: 50292045, dtype: float64

## sanity check